<a href="https://colab.research.google.com/github/yooongZa/AIFFEL_Quest_EPA/blob/main/0904_News_Bot_Projct.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 뉴스 요약봇 만들기 [프로젝트]

_이 노트북은 LMS에서 내보냈습니다. 영상·퀴즈는 학습 참고용으로 마크다운으로 변환되었습니다._

## 1. 뉴스기사 요약해보기

> 💡 **GPU 런타임 사용을 권장합니다.**

새로운 데이터셋에 대해서 추상적 요약과 추출적 요약을 모두 해보는 시간을 가져봐요.

먼저 주요 라이브러리 버전을 확인해 보죠.

In [8]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

DRIVE_DATA_DIR = Path(
    "/content/drive/MyDrive/Aiffel_EPA/news_summarization/data"
)
DRIVE_DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Google Drive 저장 경로: {DRIVE_DATA_DIR}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive 저장 경로: /content/drive/MyDrive/Aiffel_EPA/news_summarization/data


In [9]:
!pip install --upgrade summa
!pip install --upgrade nltk #3.10.3 / 3.9.1 상관없습니다.

In [10]:
from importlib.metadata import version
import nltk
import torch
import summa
import pandas as pd

print(nltk.__version__)
print(torch.__version__)
print(pd.__version__)
print(version('summa'))

3.10.3
2.11.0+cu128
2.2.3
1.2.0


## Step 1. 데이터 수집하기

데이터는 아래 링크에 있는 뉴스 기사 데이터(`news_summary_more.csv`)를 사용하세요.
- [sunnysai12345/News_Summary](https://github.com/sunnysai12345/News_Summary)<br>

아래의 코드로 데이터를 다운로드할 수 있어요.

In [11]:
from urllib.request import urlretrieve
from uuid import uuid4
import os

news_url = (
    "https://raw.githubusercontent.com/sunnysai12345/"
    "News_Summary/master/news_summary_more.csv"
)

news_csv_path = DRIVE_DATA_DIR / "news_summary_more.csv"

# Drive에 파일이 없을 때만 다운로드합니다.
if news_csv_path.is_file() and news_csv_path.stat().st_size > 0:
    print(f"기존 CSV를 사용합니다: {news_csv_path}")

else:
    temp_path = (
        DRIVE_DATA_DIR
        / f".news_summary_more.csv.{uuid4().hex}.part"
    )

    try:
        urlretrieve(news_url, temp_path)

        if temp_path.stat().st_size == 0:
            raise IOError("CSV 다운로드에 실패했습니다.")

        os.replace(temp_path, news_csv_path)
    finally:
        temp_path.unlink(missing_ok=True)

# Google Drive에서 데이터 불러오기
data = pd.read_csv(
    news_csv_path,
    encoding="iso-8859-1",
)

required_columns = {"text", "headlines"}
missing_columns = required_columns.difference(data.columns)

if missing_columns:
    raise ValueError(
        f"필수 column(열)이 없습니다: {sorted(missing_columns)}"
    )

print(f"불러오기 완료: {len(data):,}개 sample(샘플)")

기존 CSV를 사용합니다: /content/drive/MyDrive/Aiffel_EPA/news_summarization/data/news_summary_more.csv
불러오기 완료: 98,401개 sample(샘플)


In [12]:
data.sample(10)

,headlines,text
48604,Congress manufacturing issue by asking Rafale ...,Reacting to the Congress' demand that the gove...
43293,Jayasuriya deletes tweet terming B'desh conduc...,Following the on-field and off-field controver...
83945,I don't react to trolls as they don't pay my b...,Actress Nargis Fakhri has said she doesn't rea...
23838,Tesla develops limited edition surfboard price...,Elon Musk-led Tesla has developed a limited ed...
47556,PNB declares Gitanjali Group a fraud over PNB ...,Punjab National Bank has reportedly declared j...
19047,Trinamool Congress activist shot dead in West ...,A Trinamool Congress activist was shot dead by...
66648,Researchers to revisit Harappan technology for...,A group of researchers from across India will ...
4718,Thailand 1st Southeast Asian nation to legalis...,Thailand has become the first Southeast Asian ...
14631,Kerala Blasters wear special jersey to honour ...,Players of the ISL side Kerala Blasters wore j...
84033,Gautam Gambhir welcomes his second baby girl,Indian cricketer Gautam Gambhir on Wednesday w...


이 데이터는 기사의 본문에 해당되는 text와 headlines 두 가지 열로 구성되어 있습니다.<br>
추상적 요약을 하는 경우에는 text를 본문, headlines를 이미 요약된 데이터로 삼아서 모델을 학습할 수 있어요. 추출적 요약을 하는 경우에는 오직 text열만을 사용하세요.

In [13]:
import shutil
from uuid import uuid4

source_path = Path("news_summary_more.csv")
drive_path = DRIVE_DATA_DIR / source_path.name

if not source_path.is_file():
    raise FileNotFoundError("먼저 데이터 다운로드 셀을 실행해 주세요.")

if drive_path.exists():
    print(f"기존 Drive 파일을 보존합니다: {drive_path}")
else:
    temp_path = DRIVE_DATA_DIR / f".{source_path.name}.{uuid4().hex}.part"

    try:
        shutil.copy2(source_path, temp_path)

        if temp_path.stat().st_size != source_path.stat().st_size:
            raise IOError("임시 복사본의 크기 검증에 실패했습니다.")

        temp_path.replace(drive_path)
    finally:
        temp_path.unlink(missing_ok=True)

print(f"저장 완료: {drive_path}")

기존 Drive 파일을 보존합니다: /content/drive/MyDrive/Aiffel_EPA/news_summarization/data/news_summary_more.csv
저장 완료: /content/drive/MyDrive/Aiffel_EPA/news_summarization/data/news_summary_more.csv


In [14]:
import pandas as pd

drive_path = DRIVE_DATA_DIR / "news_summary_more.csv"

if not drive_path.is_file():
    raise FileNotFoundError(f"Drive 파일이 없습니다: {drive_path}")

data = pd.read_csv(drive_path, encoding="iso-8859-1")

required_columns = {"text", "headlines"}
missing_columns = required_columns.difference(data.columns)

if missing_columns:
    raise ValueError(f"필수 column(열)이 없습니다: {sorted(missing_columns)}")

print(f"불러오기 완료: {len(data):,}개 sample(샘플)")

불러오기 완료: 98,401개 sample(샘플)


## Step 2. 데이터 전처리하기 (추상적 요약)

실습에서 사용된 전처리를 참고하여 각자 필요하다고 생각하는 전처리를 추가 사용하여 텍스트를 정규화 또는 정제해 보세요. 만약, 불용어 제거를 선택한다면 상대적으로 길이가 짧은 요약 데이터에 대해서도 불용어를 제거하는 것이 좋을지 고민해 보세요.

## Step 3. 어텐션 메커니즘 사용하기 (추상적 요약)

일반적인 seq2seq보다는 어텐션 메커니즘을 사용한 seq2seq를 사용하는 것이 더 나은 성능을 얻을 수 있어요. 실습 내용을 참고하여 어텐션 메커니즘을 사용한 seq2seq를 설계해 보세요.

## Step 4. 실제 결과와 요약문 비교하기 (추상적 요약)

원래의 요약문(headlines 열)과 학습을 통해 얻은 추상적 요약의 결과를 비교해 보세요.

## Step 5. Summa를 이용해서 추출적 요약해보기

## 프로젝트 제출

## 루브릭

- Abstractive 모델 구성을 위한 텍스트 전처리 단계가 체계적으로 진행되었다.
    - 분석단계, 정제단계, 정규화와 불용어 제거, 데이터셋 분리, 인코딩 과정이 빠짐없이 체계적으로 진행되었다.
- 텍스트 요약모델이 성공적으로 학습되었음을 확인하였다.
    - 모델 학습이 진행되면서 train loss와 validation loss가 감소하는 경향을 그래프를 통해 확인했으며, 실제 요약문에 있는 핵심 단어들이 요약 문장 안에 포함되었다.
- Extractive 요약을 시도해 보고 Abstractive 요약 결과과 함께 비교해 보았다.
    - 두 요약 결과를 문법완성도 측면과 핵심단어 포함 측면으로 나누어 비교하고 분석 결과를 표로 정리하여 제시하였다.

> **[과제] 과제 제출**
>
> 노트북 파일(.ipynb)로 과제를 수행 후 출력이 저장된 상태의 깃헙 파일 링크를 입력해 주세요.
>
> 제출은 LMS 레슨 화면의 과제 카드에서 GitHub 링크로 합니다(이 파일에 적으면 제출되지 않습니다).